# TransformerEncoder

现在将各种组件组合起来形成TransformerEncoderBlock, 然后再将TransformerEncoderBlock堆叠N层组成TransformerEncoder.

由于组合和堆叠导致内部的注意力分数更加抽象和复杂, 注意力图已经没有必要保留, 所以去掉了 `attn_w`.

现在离散的token输入入口变为了TransformerEncoder, 需要把MultiHeadAttention里的embedding移到TransformerEncoder.

实验部分:

之前我们都是处理的定长序列, 我们下面要处理变长的序列, 所以要实现掩码机制, 以适配真实的环境.

## 组件

包括MHA多头注意力, FFN前馈神经网络, LayerNorm层归一化, Resnet残差连接, Dropout.

新增的组件已经在序言介绍过, 这里再回顾一下.

### 多头注意力

我们前面一直在研究的就是多头注意力. 主要作用为解决 token 跨位置依赖, 让每个位置能动态聚合全局上下文.

### 前馈神经网络

解决位置内特征变换，没有它注意力层更像线性加权汇总，表达力不足.

### 归一化

控制特征尺度漂移, 稳定梯度与优化过程. 由于输入的 token 数量不定, 所以用层归一化对每个 token 自己的特征维度做归一化.

### 残差连接

提供短路径, 缓解深层网络梯度衰减与信息退化.

### Dropout

随机给一部分神经元置零, 抑制共适应, 降低过拟合风险.

> 本文以结构机理解释为主, 不做严格消融; 结论属于理论与经验共识.



In [ ]:
import torch
from torch import nn

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout): # 由于现在离散的token输入入口变为了TransformerEncoderBlock, 需要把MultiHeadAttention里的embedding移到block
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)

        self.dropout = nn.Dropout(dropout)

        self.out = nn.Linear(d_model, d_model) # 注意这里的输出维度改为了d_model, 因为多头注意力的输出维度是d_model, 而不是output_dim

    def _rotate_half(self, x):
        # x: (batch, num_heads, seq_len, d_k)
        x1 = x[..., ::2]
        x2 = x[..., 1::2]
        return torch.stack((-x2, x1), dim=-1).flatten(-2)

    def _rope(self, q, k, base=10000):
        # q, k: (batch, num_heads, seq_len, d_k)
        seq_len = q.size(-2)
        freqs = torch.arange(0, self.d_k, 2, device=q.device) / self.d_k
        freqs = base ** -freqs
        angles = torch.arange(seq_len, device=q.device)[:, None] * freqs[None, :]
        sin = angles.sin().repeat_interleave(2, dim=-1)[None, None]
        cos = angles.cos().repeat_interleave(2, dim=-1)[None, None]
        return q * cos + self._rotate_half(q) * sin, \
              k * cos + self._rotate_half(k) * sin

    def forward(self, x, attn_mask=None):
        """
        x: (batch, seq_len, d_model)
        """
        batch_size = x.size(0)
        Q = self.w_q(x).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.w_k(x).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.w_v(x).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)

        Q, K = self._rope(Q, K)
        score = Q @ K.transpose(-2, -1) / self.d_k ** 0.5
        
        if attn_mask is not None:
            key_mask = attn_mask[:, None, None, :]
            score = score.masked_fill(key_mask == 0, -1e9)
        attn_w = torch.softmax(score, dim=-1)
        
        output = self.dropout(attn_w) @ V # 第一处Dropout层
        output = output.transpose(1, 2).reshape(batch_size, -1, self.d_model)
        output = self.out(output)
        return output


class TransformerEncoderBlock(nn.Module): # 新增block
    def __init__(self, ffn_dim, d_model, num_heads, dropout):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ffn_dim),
            nn.ReLU(),
            nn.Dropout(dropout), # 第二处Dropout层
            nn.Linear(ffn_dim, d_model),
        )
        self.dropout = nn.Dropout(dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, attn_mask=None):
        x_norm = self.norm1(x)
        x = x + self.dropout(self.attn(x_norm, attn_mask=attn_mask)) # 第三处Dropout层
        x_norm = self.norm2(x)
        x = x + self.dropout(self.ffn(x_norm)) # 第四处Dropout层
        return x


class TransformerEncoder(nn.Module): # 新增encoder
    def __init__(self, vocab_size, output_dim, ffn_dim, d_model, num_heads, num_layers, dropout=0.1, padding_idx=None): # 将embedding放在最外层，作为离散token的输入入口
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=padding_idx) # 让embedding层支持padding_idx，这样在计算注意力时就能正确地mask掉padding部分
        self.layers = nn.ModuleList([ # 通过ModuleList堆叠多个block
            TransformerEncoderBlock(ffn_dim, d_model, num_heads, dropout)
            for _ in range(num_layers)
        ])
        self.out = nn.Linear(d_model, output_dim)

    def forward(self, x, attn_mask=None):
        x = self.embedding(x)          # (batch, seq_len, d_model)
        for layer in self.layers:
            x = layer(x, attn_mask=attn_mask)
        return self.out(x)


TransformerEncoderBlock里的forward有两种写法, 一种是上面的Pre-Norm写法.

一种是下面的Transformer论文的经典写法Post-Norm, 先做注意力/FFN, 并残差连接, 再norm.

```python
def forward(self, x):
        x = x + self.attn(x) # (batch, seq_len, output_dim) 先注意力并残差
        x = self.norm1(x) # 再层归一化
        x = x + self.ffn(x) # (batch, seq_len, d_model) 先FFN并残差
        x = self.norm2(x) # 再层归一化
        return x
```
一般来说, Pre-Norm写法对于深层网络训练更稳定, 且是现在大模型的主流写法, 所以我们用这种.

## 实验1

### 数据集构建

这次我们不用自己造的toy tasks, 为了验证我们做的encoder模块在实际任务效果如何, 这次上经典的SST2数据集, 数据集每个样本包含英文影评和对应的情感标签, 1表示正向情感(好评), 0表示负向情感(差评).

为了更方便加载数据集, torch在`torch.utils.data`提供了`DataLoader`功能, 它有按照设定的`batch_size`堆叠数据, 打乱顺序等功能. 数据集我们使用 Hugging Face 的`datasets`. 由于句子长度不一致, 批处理需要对齐, 所以我们使用`transformers.DataCollatorWithPadding`进行动态填充.

英文影评需要将其token化, 即将原始文本分词后映射到token id, 以让嵌入层正常工作. 为了简化这个过程, 我们引入 `transformers.AutoTokenizer` 自动完成分词与编码, 并选择 `bert-base-uncased` 分词模型, 空白填充使用 `tokenizer` 的 `pad_token_id` (在`bert-base-uncased`模型中为0).

In [7]:
from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, DataCollatorWithPadding

batch_size = 128
dataset = load_dataset("stanfordnlp/sst2")
print(dataset)
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
print("vocab_size: ", tokenizer.vocab_size)
print("pad_token_id: ", tokenizer.pad_token_id)

def tokenize_fn(batch):
    return tokenizer(batch["sentence"], truncation=True)

# 分词
tokenized_dataset = dataset.map(tokenize_fn, batched=True)

# 改列名，去掉原始文本
tokenized_dataset = tokenized_dataset.rename_column("label", "labels")
tokenized_dataset = tokenized_dataset.remove_columns(["sentence", "idx"])

# 输出改成 torch tensor
tokenized_dataset.set_format("torch")

collate_fn = DataCollatorWithPadding(tokenizer=tokenizer)

train_dataloader = DataLoader(tokenized_dataset["train"], batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
test_dataloader = DataLoader(tokenized_dataset["test"], batch_size=batch_size, collate_fn=collate_fn)
val_dataloader = DataLoader(tokenized_dataset["validation"], batch_size=batch_size, collate_fn=collate_fn)


DatasetDict({
    train: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 872
    })
    test: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 1821
    })
})
vocab_size:  30522
pad_token_id:  0


## 2. 模型训练

这里解释一下为什么模型输出要加`[:, 0, 0]`, 在句子分类任务中, 一般会在输入序列开头加一个 `cls_token` 特殊token用于汇总这一句话的信息作为整句表示, 我们使用的 tokenizer 已经自动加上了, 所以我们只需要取第0个位置的分类logits即可.

超参数设置为:
| batch_size | vocab_size | d_model | num_heads | lr | epochs | train_steps | device |
| - | - | - | - | - | - | - | - |
| 128 | 30522 |256 | 8 | 5e-4 | 10 | 526 | cuda |

In [8]:
from sklearn.metrics import f1_score

epochs = 10
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = TransformerEncoder(vocab_size=tokenizer.vocab_size, output_dim=1, ffn_dim=512, d_model=256,
                           num_heads=8, num_layers=4, padding_idx=tokenizer.pad_token_id).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4)
criterion = nn.BCEWithLogitsLoss().to(device) # 二分类交叉熵损失

for i in range(epochs):
    model.train()
    for idx, data in enumerate(train_dataloader):
        input_ids = data["input_ids"].to(device)
        attn_mask = data["attention_mask"].to(device)
        labels = data["labels"].float().to(device)
        optimizer.zero_grad()
        pred = model(input_ids, attn_mask)[:, 0, 0]
        loss = criterion(pred, labels)
        loss.backward()
        optimizer.step()
        if (idx+1) % 100 == 0:
            print(f"Epoch {i+1}, Step {idx+1}, Loss: {loss.item():.4f}")
    
    model.eval()
    with torch.no_grad():
        all_preds = []
        all_labels = []
        for data in val_dataloader:
            input_ids = data["input_ids"].to(device)
            attn_mask = data["attention_mask"].to(device)
            labels = data["labels"].float().to(device)
            pred = model(input_ids, attn_mask)[:, 0, 0]
            pred_labels = (pred > 0).float()
            
            all_preds.extend(pred_labels.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
        
        val_f1 = f1_score(all_labels, all_preds)
        print(f"Validation F1 Score: {val_f1:.4f}")
        
    print("-"*30)

Epoch 1, Step 100, Loss: 0.4933
Epoch 1, Step 200, Loss: 0.4816
Epoch 1, Step 300, Loss: 0.3113
Epoch 1, Step 400, Loss: 0.3382
Epoch 1, Step 500, Loss: 0.3382
Validation F1 Score: 0.7960
------------------------------
Epoch 2, Step 100, Loss: 0.2187
Epoch 2, Step 200, Loss: 0.2219
Epoch 2, Step 300, Loss: 0.4077
Epoch 2, Step 400, Loss: 0.1736
Epoch 2, Step 500, Loss: 0.1928
Validation F1 Score: 0.8244
------------------------------
Epoch 3, Step 100, Loss: 0.1314
Epoch 3, Step 200, Loss: 0.1729
Epoch 3, Step 300, Loss: 0.2256
Epoch 3, Step 400, Loss: 0.2215
Epoch 3, Step 500, Loss: 0.1522
Validation F1 Score: 0.8122
------------------------------
Epoch 4, Step 100, Loss: 0.2059
Epoch 4, Step 200, Loss: 0.0693
Epoch 4, Step 300, Loss: 0.1377
Epoch 4, Step 400, Loss: 0.1337
Epoch 4, Step 500, Loss: 0.1110
Validation F1 Score: 0.8039
------------------------------
Epoch 5, Step 100, Loss: 0.0699
Epoch 5, Step 200, Loss: 0.1255
Epoch 5, Step 300, Loss: 0.1114
Epoch 5, Step 400, Loss: 0.1

## 3. 评估实验

现在在验证集上用`classification_report`验证效果.

最终F1分数能达到0.8左右, 效果不错, 足以证明我们的模型是有效的.

In [9]:
from sklearn.metrics import classification_report
model.eval()
with torch.no_grad():
    all_preds = []
    all_labels = []
    for data in val_dataloader:
        input_ids = data["input_ids"].to(device)
        attn_mask = data["attention_mask"].to(device)
        labels = data["labels"].float().to(device)
        pred = model(input_ids, attn_mask)[:, 0, 0]
        pred_labels = (pred > 0).float()
        
        all_preds.extend(pred_labels.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    
    print(classification_report(all_labels, all_preds, digits=4))

              precision    recall  f1-score   support

         0.0     0.8202    0.7780    0.7986       428
         1.0     0.7961    0.8356    0.8154       444

    accuracy                         0.8073       872
   macro avg     0.8082    0.8068    0.8070       872
weighted avg     0.8079    0.8073    0.8071       872

